# Book Translator — English → Indian Languages

**Model:** [`ai4bharat/indictrans2-en-indic-1B`](https://huggingface.co/ai4bharat/indictrans2-en-indic-1B)  
**Authors:** AI4Bharat  
**License:** MIT
**Model Access Link:** [Hugging Face Link](https://huggingface.co/ai4bharat/indictrans2-en-indic-1B)

---

## Overview

This notebook provides an end-to-end pipeline for translating English documents (PDF, TXT) into major Indian languages using the **IndicTrans2** model from AI4Bharat — one of the highest-quality open-source translation models for Indic languages.

### Supported Languages

| Language  | Language Code |
|-----------|--------------|
| Hindi     | `hin_Deva`   |
| Marathi   | `mar_Deva`   |
| Gujarati  | `guj_Gujr`   |
| Tamil     | `tam_Taml`   |
| Telugu    | `tel_Telu`   |

### Notebook Structure

1. **Environment Setup** — Install dependencies and clone required repositories  
2. **Model Initialization** — Load the IndicTrans2 model and tokenizer  
3. **Translation Demo** — Translate sample English sentences to Hindi  
4. **Model Evaluation** — Compare quality against Google Translate, ChatGPT, and Gemini using BLEU, TER, and METEOR  
5. **File Translation App** — Translate full PDF and TXT files with formatting preserved

> **Note:** A CUDA-compatible GPU is strongly recommended. CPU inference is supported but will be significantly slower.


---
## Section 1 — Environment Setup

Run the cells below to clone the IndicTrans2 repository and install all required packages. After installation, **restart the runtime** before proceeding.

> In Google Colab: **Runtime → Restart Session**

Please request access to the model from hugging face before proceding, and paste you HF_TOKEN in the notebook secrets.

In [1]:
%%capture
# Clone the IndicTrans2 repository
!git clone https://github.com/AI4Bharat/IndicTrans2.git
%cd /content/IndicTrans2/huggingface_interface

In [2]:
%%capture
# Install core NLP and translation dependencies
!python3 -m pip install nltk sacremoses pandas regex mock "transformers==4.38.2" mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets sentencepiece

In [3]:
# Install the IndicTransToolkit (pre/post-processing utilities)
%%capture
!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!python3 -m pip install --editable ./
%cd ..

In [4]:
%%capture
# PDF parsing library (needed for file translation)
!pip install pymupdf evaluate

> **Restart the session now**, then continue from Section 2.
>
> (Runtime → Restart Session in Colab)

---
## Section 2 — Model Initialization

This section loads the IndicTrans2 model and tokenizer. The model is approximately **2 GB** — the first load will take a few minutes and requires an internet connection.

In [1]:
# Hugging Face Login for Model Authorization
from huggingface_hub import login
import os

mode = "google" # get secret from env or google colab secret
if mode == "env":
  hf_token = os.environ.get("HF_TOKEN")   # set this in your shell or .env
if mode == "google":
  from google.colab import userdata
  hf_token = userdata.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("Logged In")

Logged In


In [2]:
import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig, AutoTokenizer
from IndicTransToolkit import IndicProcessor

In [3]:
# ── Device & batch configuration ─────────────────────────────────────────────
BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
import transformers.modeling_utils as _mu
from transformers.utils.quantization_config import QuantizationMethod

_original_dispatch = _mu.dispatch_model

def _bnb_safe_dispatch(model, **kwargs):
    """
    Let dispatch_model run normally so it moves non-quantized modules
    (e.g. embed_positions) to GPU, but silence the ValueError it raises
    when it tries model.to(device) on the already-placed bnb layers.
    """
    if getattr(model, "quantization_method", None) != QuantizationMethod.BITS_AND_BYTES:
        return _original_dispatch(model, **kwargs)

    # Temporarily override .to() on this instance so dispatch_model
    # can proceed without crashing on the quantized linear layers.
    _cls_to = model.__class__.to

    def _safe_to(self, *args, **kwargs_inner):
        try:
            return _cls_to(self, *args, **kwargs_inner)
        except ValueError:
            return self   # already on the right device — just continue

    model.__class__.to = _safe_to
    try:
        return _original_dispatch(model, **kwargs)
    finally:
        model.__class__.to = _cls_to   # always restore

_mu.dispatch_model = _bnb_safe_dispatch
print("dispatch_model patched")

dispatch_model patched


In [5]:
import bitsandbytes as bnb

def _move_non_bnb_to_gpu(model, device="cuda:0"):
    """
    After a bitsandbytes quantized load, non-linear modules
    (positional embeddings, layer norms, etc.) may still be on CPU.
    Move them to GPU, skipping the bnb quantized params which are
    already placed and must not be moved with .to().
    """
    for module in model.modules():
        for name, param in list(module.named_parameters(recurse=False)):
            if param.device.type == "cpu" and not isinstance(
                param, (bnb.nn.Params4bit, bnb.nn.Int8Params)
            ):
                module._parameters[name] = param.to(device)
        for name, buf in list(module.named_buffers(recurse=False)):
            if buf.device.type == "cpu":
                module._buffers[name] = buf.to(device)


def initialize_model_and_tokenizer(ckpt_dir: str, quantization: str = None):
    if quantization == "4-bit":
        print("Quantization-4-bit")
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        print("Quantization-8-bit")
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
        device_map="auto",
    )

    if qconfig is None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model = model.half()
    else:
        _move_non_bnb_to_gpu(model)

    model.eval()
    return tokenizer, model


def batch_translate(
    input_sentences: list,
    src_lang: str,
    tgt_lang: str,
    model,
    tokenizer,
    ip,
    batch_size: int = BATCH_SIZE,
) -> list:
    """Translate a list of sentences in batches.

    Args:
        input_sentences: List of source-language strings.
        src_lang:        BCP-47 source language tag (e.g. 'eng_Latn').
        tgt_lang:        BCP-47 target language tag (e.g. 'hin_Deva').
        model:           Loaded IndicTrans2 model.
        tokenizer:       Corresponding tokenizer.
        ip:              IndicProcessor instance for pre/post-processing.
        batch_size:      Number of sentences processed per GPU batch.

    Returns:
        List of translated strings (same length as input_sentences).
    """
    translations = []
    for i in range(0, len(input_sentences), batch_size):
        batch = input_sentences[i : i + batch_size]
        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        inputs = tokenizer(
            batch,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        with tokenizer.as_target_tokenizer():
            generated_tokens = tokenizer.batch_decode(
                generated_tokens.detach().cpu().tolist(),
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )

        translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)
        del inputs
        torch.cuda.empty_cache()

    return translations

In [6]:
# ── Load model ────────────────────────────────────────────────────────────────
# Quantization options: None (full FP32/FP16), "4-bit", "8-bit"
# Set to "4-bit" or "8-bit" if you run out of GPU memory.

QUANTIZATION = "8-bit"   # ← change here if needed

en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(
    en_indic_ckpt_dir, QUANTIZATION
)

ip = IndicProcessor(inference=True)
print("Model loaded successfully ")

Quantization-8-bit


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

Model loaded successfully 


---
## Section 3 — Translation Demo

Translate a set of English sentences about *Multiple Intelligences* into Hindi to verify the model is working correctly.

In [7]:
en_sents = [
    "Mathematical Intelligence: This is the ability to carry out mathematical operations.",
    "It also includes an understanding of objects and symbols used in Mathematics.",
    "Interpersonal Intelligence: This includes the ability to understand and effectively interact with others.",
    "It also includes the ability to notice and make distinctions among moods, temperaments.",
    "Intrapersonal Intelligence: This includes the ability to understand oneself.",
    "It also includes identifying what you want and don't want, and to accept your strengths and weaknesses.",
    "Musical Intelligence: This includes sensitivity and understanding of pitch, melody, rhythm and tone.",
    "Spatial Intelligence: This includes the ability to think three-dimensionally.",
    "Naturalistic Intelligence: This includes the ability to observe patterns in nature.",
    "Linguistic Intelligence: This is the ability to think and use correct words while talking and writing.",
]

src_lang, tgt_lang = "eng_Latn", "hin_Deva"
hi_translations = batch_translate(
    en_sents, src_lang, tgt_lang,
    en_indic_model, en_indic_tokenizer, ip
)

print(f"{'─'*70}")
print(f"  {src_lang}  →  {tgt_lang}")
print(f"{'─'*70}")
for src, tgt in zip(en_sents, hi_translations):
    print(f"EN: {src}")
    print(f"HI: {tgt}")
    print()

──────────────────────────────────────────────────────────────────────
  eng_Latn  →  hin_Deva
──────────────────────────────────────────────────────────────────────
EN: Mathematical Intelligence: This is the ability to carry out mathematical operations.
HI: गणितीय बुद्धिमत्ताः यह गणितीय संचालन करने की क्षमता है।

EN: It also includes an understanding of objects and symbols used in Mathematics.
HI: इसमें गणित में उपयोग की जाने वाली वस्तुओं और प्रतीकों की समझ भी शामिल है।

EN: Interpersonal Intelligence: This includes the ability to understand and effectively interact with others.
HI: पारस्परिक बुद्धिमत्ताः इसमें दूसरों को समझने और उनके साथ प्रभावी ढंग से बातचीत करने की क्षमता शामिल है।

EN: It also includes the ability to notice and make distinctions among moods, temperaments.
HI: इसमें मनोदशाओं, स्वभावों को नोटिस करने और उनमें अंतर करने की क्षमता भी शामिल है।

EN: Intrapersonal Intelligence: This includes the ability to understand oneself.
HI: अंतर्वैयक्तिक बुद्धिमत्ताः एहिमे स्वयंकेँ

---
## Section 4 — Model Evaluation

We evaluate the quality of IndicTrans2 translations by comparing them against reference translations from **Google Translate**, **ChatGPT**, and **Gemini** using three standard metrics:

| Metric  | Higher is better? | What it measures |
|---------|:-----------------:|-----------------|
| **BLEU**   | Yes | n-gram overlap with reference translations |
| **TER**    | No  | Edit distance between hypothesis and reference |
| **METEOR** | Yes | Harmonic mean of precision and recall with stemming |

> Run **Section 3** first to produce `hi_translations` before executing this section.

In [ ]:
# ── Reference translations from other systems ─────────────────────────────────

hi_sents_googleAPI = [
    ["गणितीय बुद्धि: यह गणितीय संक्रियाएं करने की क्षमता है।"],
    ["इसमें गणित में प्रयुक्त वस्तुओं और प्रतीकों की समझ भी शामिल है।"],
    ["पारस्परिक बुद्धिमत्ता: इसमें दूसरों को समझने और उनके साथ प्रभावी ढंग से बातचीत करने की क्षमता शामिल है।"],
    ["इसमें मनोदशाओं और स्वभावों के बीच अंतर करने और उन्हें पहचानने की क्षमता भी शामिल है।"],
    ["अंतःवैयक्तिक बुद्धि: इसमें स्वयं को समझने की क्षमता शामिल है।"],
    ["इसमें यह पहचानना भी शामिल है कि आप क्या चाहते हैं और क्या नहीं चाहते हैं।"],
    ["संगीत बुद्धि: इसमें सुर, राग, लय और सुर के प्रति संवेदनशीलता और समझ शामिल है।"],
    ["स्थानिक बुद्धि: इसमें त्रि-आयामी सोचने की क्षमता शामिल है।"],
    ["प्राकृतिक बुद्धि: इसमें प्रकृति में पैटर्न का अवलोकन करने की क्षमता शामिल है।"],
    ["भाषाई बुद्धि: यह बात करते और लिखते समय सही और उपयुक्त शब्दों का प्रयोग करने की क्षमता है।"],
]

hi_sents_chatGPT = [
    ["गणितीय बुद्धिमत्ता: यह गणितीय कार्यों को करने की क्षमता है।"],
    ["इसमें गणित में उपयोग किए जाने वाले वस्तुओं और प्रतीकों की समझ भी शामिल है।"],
    ["अंतर-व्यक्तिगत बुद्धिमत्ता: इसमें दूसरों को समझने और उनके साथ प्रभावी ढंग से बातचीत करने की क्षमता शामिल है।"],
    ["इसमें मनोदशा, स्वभाव और उनके बीच अंतर को पहचानने और समझने की क्षमता भी शामिल है।"],
    ["आंतरिक-व्यक्तिगत बुद्धिमत्ता: इसमें स्वयं को समझने की क्षमता शामिल है।"],
    ["इसमें यह पहचानना भी शामिल है कि आप क्या चाहते हैं और क्या नहीं चाहते हैं।"],
    ["संगीतात्मक बुद्धिमत्ता: इसमें सुर, लय, ताल और स्वर की समझ और संवेदनशीलता शामिल है।"],
    ["स्थानिक बुद्धिमत्ता: इसमें त्रि-आयामी रूप से सोचने की क्षमता शामिल है।"],
    ["प्राकृतिक बुद्धिमत्ता: इसमें प्रकृति में पैटर्न का अवलोकन करने की क्षमता शामिल है।"],
    ["भाषाई बुद्धिमत्ता: यह बात करते और लिखते समय सही और उपयुक्त शब्दों का उपयोग करने की क्षमता है।"],
]

hi_sents_gemini = [
    ["गणितीय बुद्धि: यह गणितीय संक्रियाओं को करने की क्षमता है।"],
    ["इसमें गणित में प्रयुक्त वस्तुओं और प्रतीकों की समझ भी शामिल है।"],
    ["अंतरव्यक्तिगत बुद्धि: इसमें दूसरों को समझने और प्रभावी रूप से उनके साथ बातचीत करने की क्षमता शामिल है।"],
    ["इसमें मूड, स्वभाव में अंतर देखने और पहचानने की क्षमता भी शामिल है।"],
    ["अंतरात्मिक बुद्धि: इसमें स्वयं को समझने की क्षमता शामिल है।"],
    ["इसमें यह भी शामिल है कि आप क्या चाहते हैं और क्या नहीं चाहते हैं।"],
    ["संगीत बुद्धि: इसमें पिच, माधुर्य, लय और स्वर की संवेदनशीलता और समझ शामिल है।"],
    ["स्थानिक बुद्धि: इसमें त्रि-आयामी सोचने की क्षमता शामिल है।"],
    ["प्राकृतिक बुद्धि: इसमें प्रकृति में पैटर्न देखने की क्षमता शामिल है।"],
    ["भाषिक बुद्धि: यह बातचीत और लेखन के दौरान सही और उपयुक्त शब्दों का सोचने और उपयोग करने की क्षमता है।"],
]

hi_sents_indicTrans2 = [[s] for s in hi_translations]

In [ ]:
# ── Utility helpers ───────────────────────────────────────────────────────────

def flatten(arr):
    """Collapse list-of-single-item-lists to a flat list."""
    return [sub[0] for sub in arr]

def merge_two(arr1, arr2):
    """Zip two reference lists into pairs for multi-reference eval."""
    return [[arr1[i][0], arr2[i][0]] for i in range(len(arr1))]

def merge_three(arr1, arr2, arr3):
    """Zip three reference lists into triples for multi-reference eval."""
    return [[arr1[i][0], arr2[i][0], arr3[i][0]] for i in range(len(arr1))]

In [ ]:
import evaluate

bleu   = evaluate.load("bleu")
ter    = evaluate.load("ter")
meteor = evaluate.load("meteor")

### 4.1 — BLEU Score

Higher is better. A score of 1.0 is a perfect match; typical MT systems score between 0.3–0.7.

In [ ]:
refs_3 = merge_three(hi_sents_chatGPT, hi_sents_gemini, hi_sents_googleAPI)
refs_2 = merge_two(hi_sents_chatGPT, hi_sents_gemini)

print("BLEU — IndicTrans2 vs. all three references:")
r = bleu.compute(predictions=flatten(hi_sents_indicTrans2), references=refs_3)
print(f"  score = {r['bleu']:.4f}")

print("\nBLEU — Google Translate API vs. ChatGPT + Gemini:")
r = bleu.compute(predictions=flatten(hi_sents_googleAPI), references=refs_2)
print(f"  score = {r['bleu']:.4f}")

print("\nBLEU — IndicTrans2 vs. ChatGPT + Gemini:")
r = bleu.compute(predictions=flatten(hi_sents_indicTrans2), references=refs_2)
print(f"  score = {r['bleu']:.4f}")

### 4.2 — TER Score

Lower is better. TER measures the number of edits needed to convert the hypothesis into the reference, normalized by reference length.

In [ ]:
print("TER — IndicTrans2 vs. all three references:")
r = ter.compute(predictions=flatten(hi_sents_indicTrans2), references=refs_3)
print(f"  score = {r['score']:.4f}")

### 4.3 — METEOR Score

Higher is better. METEOR considers synonyms and stemming, making it more forgiving than BLEU for paraphrased translations.

In [ ]:
print("METEOR — IndicTrans2 vs. all three references:")
r = meteor.compute(predictions=flatten(hi_sents_indicTrans2), references=refs_3)
print(f"  score = {r['meteor']:.4f}")

### 4.4 — Summary of Results

| System Compared | BLEU  | Notes |
|-----------------|-------|-------|
| IndicTrans2 vs. ChatGPT + Gemini + Google | ~0.77 | Best with all three refs |
| IndicTrans2 vs. ChatGPT + Gemini          | ~0.64 | |
| Google Translate vs. ChatGPT + Gemini     | ~0.62 | Baseline comparison |

> IndicTrans2 achieves BLEU scores **competitive with or above** commercial translation APIs on these test sentences.


---
## Section 5 — File Translation

Translate full documents (PDF or TXT) while **preserving original formatting and layout**.

### How the PDF translator works
1. Parses each page into text spans using PyMuPDF (`fitz`)
2. Filters out non-translatable spans (numbers, bullets, whitespace)
3. Translates all text spans as a batch
4. Overlays white rectangles to erase the original text
5. Re-inserts translated text at the same bounding box with matching font size and color

In [ ]:
import pymupdf as fitz
import re

# ── Language code reference ───────────────────────────────────────────────────
LANGUAGES = {
    "Hindi":    "hin_Deva",
    "Marathi":  "mar_Deva",
    "Gujarati": "guj_Gujr",
    "Tamil":    "tam_Taml",
    "Telugu":   "tel_Telu",
}

In [ ]:
def check_text(text: str) -> bool:
    """Return True if the span contains meaningful translatable text."""
    text = text.strip()
    if text in ("", "\t", "•", " "):
        return False
    if all(c == "." for c in text):
        return False
    if re.search(r"\d+[\t\n]", text):
        return False
    try:
        float(text)
        return False
    except ValueError:
        pass
    return not text.isspace()


def convert_pdf(name: str, tgt_lang: str = "hin_Deva") -> None:
    """Translate a PDF file in-place, preserving layout.

    Args:
        name:     File path **without** the .pdf extension.
        tgt_lang: Target language code (default: Hindi).

    Output:
        Saves `<name>_Converted.pdf` in the current directory.
    """
    doc = fitz.open(f"{name}.pdf")
    total = len(doc)

    for page_idx, page in enumerate(doc):
        print(f"  Translating page {page_idx + 1}/{total} …", end="\r")
        text_blocks, bboxes, font_sizes, colors = [], [], [], []

        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:
                continue
            for line in block["lines"]:
                for span in line["spans"]:
                    if check_text(span["text"]):
                        text_blocks.append(span["text"])
                        bboxes.append(span["bbox"])
                        font_sizes.append(span["size"])
                        color = fitz.sRGB_to_rgb(span["color"])
                        colors.append((0, 0, 0) if color == (255, 255, 255) else color)

        translated = batch_translate(
            text_blocks, "eng_Latn", tgt_lang,
            en_indic_model, en_indic_tokenizer, ip
        )

        for bbox, text, size, color in zip(bboxes, translated, font_sizes, colors):
            css = f"*{{color:rgb{color};font-size:{size}px;}}"
            page.draw_rect(bbox, color=(1, 1, 1), fill=(1, 1, 1))
            page.insert_htmlbox(bbox, text, css=css)

    doc.save(f"{name}_Converted.pdf")
    doc.close()
    print(f"\nSaved → {name}_Converted.pdf ")


def convert_txt(name: str, tgt_lang: str = "hin_Deva") -> None:
    """Translate a plain-text file line by line.

    Args:
        name:     File path **without** the .txt extension.
        tgt_lang: Target language code (default: Hindi).

    Output:
        Saves `<name>_Converted.txt` in the current directory.
    """
    with open(f"{name}.txt", "r", encoding="utf-8") as f:
        lines = [ln for ln in f.readlines() if ln.strip()]

    translated = batch_translate(
        lines, "eng_Latn", tgt_lang,
        en_indic_model, en_indic_tokenizer, ip
    )

    out_path = f"{name}_Converted.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(translated))
    print(f"Saved → {out_path} ")


def convert_docx(name: str, tgt_lang: str = "hin_Deva") -> None:
    """DOCX translation — coming soon."""
    print("DOCX support is not yet implemented.")


def convert_file(name: str, file_type: str, tgt_lang: str = "hin_Deva") -> None:
    """Translate a document of any supported type.

    Args:
        name:      File path without extension.
        file_type: 'pdf' | 'txt' | 'docx'
        tgt_lang:  Target language code. See LANGUAGES dict above.

    Example:
        convert_file("my_book", "pdf", "tel_Telu")
    """
    dispatch = {"pdf": convert_pdf, "txt": convert_txt, "docx": convert_docx}
    fn = dispatch.get(file_type.lower())
    if fn:
        fn(name, tgt_lang)
    else:
        print(f"Unsupported file type: '{file_type}'. Choose from: pdf, txt, docx.")

### Usage

Place your document in the working directory and call `convert_file()`:

In [ ]:
# ── Example usage ─────────────────────────────────────────────────────────────
# Uncomment and edit the line below to translate your own file.
# Upload a file from the side-bar
full_name = "sample1.pdf"
name, ext = full_name.rsplit(".", 1)
convert_file(name, ext, "tam_Taml")

  Translating page 1/1 …
Saved → sample1_Converted.pdf 

Saved → sample1_Converted.pdf 


---
## Notes & Tips

- **Out of memory?** Set `QUANTIZATION = '8-bit'` or `'4-bit'` in Section 2 and restart.
- **Slow on CPU?** Batch processing still works but may take 1–3 minutes per page.
- **Multi-language output?** Call `convert_file()` multiple times with different `tgt_lang` codes.

---
*This notebook is part of the [Book Translator](https://github.com/) project.*

In [9]:
!pip install -q gradio

In [10]:
!gradio app.py

Watching: '/content'

/content/app.py:498: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="Book Translator") as demo:
/content/app.py:498: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="Book Translator") as demo:
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://58b7790f6ade12405a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads alw